# Reconeixement de Vocals en Espanyol

## Objectiu
Classificar vocals espanyoles (A, E, I, O, U) a partir d'un segment d'àudio.

## Pipeline
1. **Imports** — llibreries necessàries
2. **Dataset** — carregar `Edu.json` + fitxer WAV i retallar cada vocal
3. **Model** — convertir cada segment d'àudio a un vector de característiques (formants F1, F2, F3) via FFT
4. **Entrenament** — construir la matriu de característiques per a tots els exemples
5. **Pèrdua / Optimitzador** — classificació per distància euclidiana entre vectors de formants
6. **Main** — provar el sistema amb una nova mostra

In [ ]:
import json
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from numpy.fft import fft
from IPython.display import Audio
from wav2vec import cutvowel, wav2vec, distancebv

### Imports
- `json` — per llegir el dataset d'anotacions (`Edu.json`)
- `numpy` — operacions numèriques i FFT
- `scipy.io` — llegir fitxers WAV (`wavfile.read`)
- `matplotlib` — visualitzar formes d'ona i espectres
- `IPython.display.Audio` — reproduir àudio inline
- `wav2vec` — mòdul propi: `cutvowel` retalla un segment, `wav2vec` extreu els tres formants principals, `distancebv` calcula distància euclidiana entre vectors

In [ ]:
WAV_FILE  = "vowels/beppo.wav"
JSON_FILE = "vowels/Edu.json"

# Carreguem les anotacions: llista de {"vocal", "start", "end"}
with open(JSON_FILE) as f:
    data = json.load(f)

# Carreguem el fitxer WAV sencer
Fs, audio_full = sio.wavfile.read(WAV_FILE)
print(f"Mostres totals: {len(audio_full)}  |  Freq. mostreig: {Fs} Hz")
print(f"Durada: {len(audio_full)/Fs:.2f} s  |  Vocals anotades: {len(data)}")

# Exemple: vocal número 30
sample = data[30]
print(f"\nExemple [{30}]: {sample}")

# Retallem el segment d'àudio corresponent
start = float(sample["start"])
end   = float(sample["end"])
cut   = audio_full[int(start * Fs): int(end * Fs), 0]   # canal esquerra (mono)

Audio(cut, rate=Fs)

### Dataset i DataLoader

`Edu.json` conté **50 entrades** (10 per cada vocal A/E/I/O/U) generades automàticament per `main.py`:
- Es sintetitza el text en castellà amb gTTS → fitxer WAV
- Es detecten segments vocalics per llindar d'energia i anàlisi de formants (Praat)
- Cada entrada guarda `vocal`, `start` i `end` en segons

Per a cada mostra:
1. Es llegeix el WAV sencer amb `scipy.io.wavfile.read`
2. Es retalla el segment `audio[start*Fs : end*Fs]`
3. El vector resultant és l'entrada al model

> El DataLoader aquí és manual (bucle sobre `data`), sense batching — el dataset és petit.

In [ ]:
# --- Visualitzem la transformació pas a pas per a un segment ---

idx = 30
Fs, cut = cutvowel(WAV_FILE, data[idx]["start"], data[idx]["end"])

# 1. Forma d'ona temporal
plt.figure(figsize=(10, 3))
plt.plot(cut)
plt.title(f"Forma d'ona — vocal '{data[idx]['vocal']}'")
plt.xlabel("Mostres"); plt.ylabel("Amplitud"); plt.tight_layout(); plt.show()

# 2. FFT completa (valors complexos)
fourierofcut = fft(cut)
Fsmall = fourierofcut[0:300]                           # conservem les 300 primeres freqs. (baixes)

# 3. Mòdul (eliminem la fase)
magnitude = np.sqrt(np.real(Fsmall)**2 + np.imag(Fsmall)**2)
magnitude[0:30] = 0                                    # eliminem DC + soroll molt baix

# 4. Filtre de mitjana mòbil (suavitzat)
lpf = 15
smoothed = np.zeros(len(magnitude) - lpf, dtype=np.float64)
for i in range(len(magnitude) - lpf):
    smoothed[i] = magnitude[i:i+lpf].sum()

plt.figure(figsize=(10, 3))
plt.plot(smoothed)
plt.title("Espectre suavitzat (mòdul FFT, freqs. baixes)")
plt.xlabel("Índex freqüència"); plt.ylabel("Energia"); plt.tight_layout(); plt.show()

# 5. Formants F1, F2, F3 via wav2vec
formants = wav2vec(cut, Fs)
print(f"Formants (Hz): F1={formants[0]:.0f}  F2={formants[1]:.0f}  F3={formants[2]:.0f}")

### Model — wav2vec (extracció de formants)

La funció `wav2vec(cut, Fs)` de `wav2vec.py` converteix un segment d'àudio en un vector de 3 valors: **[F1, F2, F3]** (els tres primers formants en Hz).

**Passos interns:**
1. **FFT** sobre el segment retallar → espectre de freqüències complex
2. **Retall** als primers 300 bins (cobreix ~0–3000 Hz a 48 kHz, on viuen els formants vocals)
3. **Mòdul** `sqrt(re² + im²)` — eliminem la fase, ens quedem l'energia per freqüència
4. **Zeros DC** — esborrem els primers 30 bins (soroll de molt baixa freqüència)
5. **Filtre de mitjana mòbil** (finestra `lpf=15`) — suavitzem pics espuris
6. **Detecció de màxims** — trobem els 3 pics dominants eliminant un entorn de ±25 bins al voltant de cada màxim per evitar detectar el mateix pic dues vegades
7. **Conversió a Hz**: `freqüència = índex × Fs / len(fft_complet)`

> Els formants vocàlics reflecteixen la forma del tracte vocal. Cada vocal espanyola té una "empremta" característica de F1 i F2:
> | Vocal | F1 (Hz) | F2 (Hz) |
> |-------|---------|---------|
> | A     | ~750    | ~1250   |
> | E     | ~500    | ~1900   |
> | I     | ~300    | ~2300   |
> | O     | ~500    | ~900    |
> | U     | ~300    | ~800    |

In [ ]:
# Construïm la matriu de característiques (train)
# Forma: N × 3  (N mostres, 3 formants per mostra)

labels  = []          # etiquetes (vocal: A/E/I/O/U)
vectors = []          # vectors de formants [F1, F2, F3]

for entry in data:
    Fs_i, cut_i = cutvowel(WAV_FILE, entry["start"], entry["end"])
    vec = wav2vec(cut_i, Fs_i)
    labels.append(entry["vocal"])
    vectors.append(vec)

X = np.array(vectors)   # shape (50, 3)
y = np.array(labels)    # shape (50,)

print(f"X shape: {X.shape}")
print(f"Classes: {np.unique(y)}")

# Visualitzem F1 vs F2 per a totes les mostres
colors = {"A": "red", "E": "blue", "I": "green", "O": "orange", "U": "purple"}
plt.figure(figsize=(8, 6))
for vocal in "AEIOU":
    mask = y == vocal
    plt.scatter(X[mask, 0], X[mask, 1], label=vocal, color=colors[vocal], alpha=0.7)
plt.xlabel("F1 (Hz)"); plt.ylabel("F2 (Hz)")
plt.title("Espai de formants F1 vs F2")
plt.legend(); plt.tight_layout(); plt.show()

### Bucle d'entrenament

No hi ha retropropagació en aquest model — el "entrenament" consisteix a **calcular el vector de formants per a cada mostra** i guardar-lo a la matriu `X`.

Cada entrada `(Fs, cut)` passa per `wav2vec` i retorna `[F1, F2, F3]`. El resultat és:
- `X` — matriu de forma `(N, 3)` amb els formants de cada mostra
- `y` — vector d'etiquetes de classe

El gràfic **F1 vs F2** mostra si les vocals son linealment separables en l'espai de formants. En espanyol, les 5 vocals estan ben distribuïdes: /i/ i /u/ baix F1, /a/ alt F1, /e/ i /o/ intermig.

In [ ]:
def predict(test_vec, X_train, y_train):
    """
    Classificador 1-NN per distància euclidiana sobre vectors de formants.
    Retorna la vocal predita i la distància al veí més proper.
    """
    distances = np.array([distancebv(test_vec, x) for x in X_train])
    nearest   = np.argmin(distances)
    return y_train[nearest], distances[nearest]


# --- Avaluació Leave-One-Out (LOO) ---
correct = 0
for i, entry in enumerate(data):
    # Màscara: tots els exemples excepte el i-èsim
    mask      = np.arange(len(data)) != i
    X_train_i = X[mask]
    y_train_i = y[mask]

    pred, dist = predict(X[i], X_train_i, y_train_i)
    if pred == y[i]:
        correct += 1
    else:
        print(f"  Error [{i}]: real={y[i]}  pred={pred}  dist={dist:.1f} Hz")

accuracy = correct / len(data) * 100
print(f"\nExactes: {correct}/{len(data)}  |  Precisió LOO: {accuracy:.1f}%")

### Pèrdua i Optimitzador — Classificació per Distància Euclidiana

No s'utilitza gradient descent. El classificador és **1-Nearest Neighbour (1-NN)**:

```
distància(v1, v2) = sqrt((F1₁-F1₂)² + (F2₁-F2₂)² + (F3₁-F3₂)²)
```

**Predicció**: donada una mostra nova, es calcula la distància al vector de cada exemple del dataset i es retorna l'etiqueta del més proper.

**Avaluació Leave-One-Out (LOO)**: per a cada mostra, s'entrena amb les restants N−1 i es prediu la mostra exclosa. Això mesura la generalització sense necessitat d'un conjunt de test separat (ideal per a datasets petits).

> **Per què no hi ha "loss" clàssica?**  
> Els formants no s'aprenen — es calculen directament de la física del so (FFT + pics espectrals). El model és completament interpretatble i no té paràmetres entrenables.

In [ ]:
# --- Prova amb una nova mostra del dataset ---
# Canvia aquest índex per qualsevol entrada de Edu.json
test_idx  = 5
test_entry = data[test_idx]

Fs_t, cut_t = cutvowel(WAV_FILE, test_entry["start"], test_entry["end"])
test_vec    = wav2vec(cut_t, Fs_t)

pred, dist = predict(test_vec, X, y)

print(f"Mostra   : idx={test_idx}  vocal real='{test_entry['vocal']}'")
print(f"Formants : F1={test_vec[0]:.0f} Hz  F2={test_vec[1]:.0f} Hz  F3={test_vec[2]:.0f} Hz")
print(f"Prediccio: '{pred}'  (distancia={dist:.1f} Hz)")
print(f"Resultat : {'CORRECTE' if pred == test_entry['vocal'] else 'ERROR'}")

Audio(cut_t, rate=Fs_t)

### Main — Prova del sistema complet

Aquí es verifica el funcionament del pipeline de principi a fi:

1. Es selecciona una entrada de `Edu.json` (canviant `test_idx`)
2. Es retalla el segment d'àudio corresponent amb `cutvowel`
3. S'extreuen els formants amb `wav2vec` → vector `[F1, F2, F3]`
4. Es classifica per distància 1-NN contra tot el dataset `X`
5. Es compara la predicció amb l'etiqueta real

**Resum del pipeline complet:**
```
text castellà
    → gTTS TTS → beppo.wav
    → detecció segments vocalics (energia + Praat)
    → Edu.json  (start, end, vocal)
    → cutvowel  → segment WAV
    → FFT + mòdul + filtre + màxims
    → [F1, F2, F3]
    → distància euclidiana 1-NN
    → vocal predita
```